# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

## 1.2 Функции

In [102]:
def evaluate_classification(y_test, y_pred_proba):
    """Оценивает результаты классификации"""
    # Кодируем строковые метки в числа
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    
    # Предсказанные классы
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Метрики
    accuracy = accuracy_score(y_test_encoded, y_pred)
    f1 = f1_score(y_test_encoded, y_pred, average='weighted')
    
    # Для многоклассовой классификации
    auc_roc = roc_auc_score(y_test_encoded, y_pred_proba, multi_class='ovr', average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"AUC-ROC: {auc_roc:.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test_encoded, y_pred, target_names=['L (падение)', 'N (нейтр)', 'H (рост)']))

In [104]:
def train_predict_rf(X_train, X_test, y_train):
    """Обучает и предсказывает Random Forest"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    model = RandomForestClassifier(n_estimators=100, random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

def train_predict_dt(X_train, X_test, y_train):
    """Обучает и предсказывает Decision Tree"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    model = DecisionTreeClassifier(random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

def train_predict_logreg(X_train, X_test, y_train):
    """Обучает и предсказывает Logistic Regression"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression(max_iter=1000, random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

def train_predict_catboost(X_train, X_test, y_train):
    """Обучает и предсказывает CatBoost"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    model = CatBoostClassifier(verbose=False, random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

In [106]:
def train_test_split_by_date(df, target_column, test_size=0.2):
    """
    Разбивает данные на train/test по дате и возвращает X, y
    
    Args:
        df: DataFrame с колонкой 'begin'
        target_column: название целевой переменной
        test_size: доля тестовых данных (0.2 = 20%)
    """
    df = df.sort_values('begin').reset_index(drop=True)
    
    # Вычисляем индекс разбиения
    split_idx = int(len(df) * (1 - test_size))
    
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    print(f"Train: {train_df['begin'].min()} - {train_df['begin'].max()} ({len(train_df)} samples)")
    print(f"Test:  {test_df['begin'].min()} - {test_df['begin'].max()} ({len(test_df)} samples)")
    
    # Удаляем колонку 'begin' и разделяем на X, y
    X_train = train_df.drop(columns=['begin', target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=['begin', target_column])
    y_test = test_df[target_column]
    
    print(f"Признаков: {X_train.shape[1]}")
    
    return X_train, X_test, y_train, y_test

# 2 Подготовка данных

## 2.0 Список тикеров

In [110]:
tickers = [
    'SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN'
]

## 2.1 Чтение

In [113]:
data_SBER = pd.read_csv("../../../data/stock_features_data/stocks_features_SBER.csv")
data_TCSG = pd.read_csv("../../../data/stock_features_data/stocks_features_TCSG.csv")
data_GAZP = pd.read_csv("../../../data/stock_features_data/stocks_features_GAZP.csv")
data_LKOH = pd.read_csv("../../../data/stock_features_data/stocks_features_LKOH.csv")
data_ROSN = pd.read_csv("../../../data/stock_features_data/stocks_features_ROSN.csv")
data_ROSN.head(2)

,begin,close,MA_90,RSI_14,RSI_90,MOM_10,ATR_14,VOLATILITY_20,VOLATILITY_50,VOLUME_RATIO_20,MACD_SIGNAL,MACD_HISTOGRAM,target_price_change,target_class
0,2022-05-31 18:00:00,377.00,385.952778,41.700213,46.109636,-6.102117,6.815141,0.177819,0.184228,0.180549,1.622548,-2.115281,-2.771883,L
1,2022-06-01 10:00:00,378.35,385.708333,43.536971,46.522581,-3.703232,6.960488,0.153935,0.177697,0.770980,1.125999,-1.986196,-1.361174,L


## 2.2 Удаление лишних признаков

In [115]:
# Для SBER
part_SBER = data_SBER[['begin', 'close', 'target_price_change']]
data_SBER.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для TCSG
part_TCSG = data_TCSG[['begin', 'close', 'target_price_change']]
data_TCSG.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для GAZP
part_GAZP = data_GAZP[['begin', 'close', 'target_price_change']]
data_GAZP.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для LKOH
part_LKOH = data_LKOH[['begin', 'close', 'target_price_change']]
data_LKOH.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для ROSN
part_ROSN = data_ROSN[['begin', 'close', 'target_price_change']]
data_ROSN.drop(['close', 'target_price_change'], axis=1, inplace=True)

## 2.3 Разделение на train/test

In [119]:
X_train_SBER, X_test_SBER, y_train_SBER, y_test_SBER = train_test_split_by_date(
    df=data_SBER, 
    target_column='target_class',
    test_size=0.2
)

X_train_TCSG, X_test_TCSG, y_train_TCSG, y_test_TCSG = train_test_split_by_date(
    df=data_TCSG, 
    target_column='target_class',
    test_size=0.2
)

X_train_GAZP, X_test_GAZP, y_train_GAZP, y_test_GAZP = train_test_split_by_date(
    df=data_GAZP, 
    target_column='target_class',
    test_size=0.2
)

X_train_LKOH, X_test_LKOH, y_train_LKOH, y_test_LKOH = train_test_split_by_date(
    df=data_LKOH, 
    target_column='target_class',
    test_size=0.2
)

X_train_ROSN, X_test_ROSN, y_train_ROSN, y_test_ROSN = train_test_split_by_date(
    df=data_ROSN, 
    target_column='target_class',
    test_size=0.2
)

Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2024-06-03 10:00:00 (2367 samples)
Test:  2024-06-03 12:00:00 - 2024-11-20 18:00:00 (592 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-03-11 14:00:00 (3427 samples)
Test:  2025-03-11 16:00:00 - 2025-09-30 18:00:00 (857 samples)
Признаков: 10


# 3 Обучение простых моделей 

## 3.1 Дерево решений

### 3.1.1 Сбер

In [124]:
y_pred_SBER = train_predict_dt(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.4131
F1-score: 0.3767
AUC-ROC: 0.5655

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.81      0.50       269
   N (нейтр)       0.67      0.30      0.41       379
    H (рост)       0.29      0.11      0.15       209

    accuracy                           0.41       857
   macro avg       0.44      0.41      0.35       857
weighted avg       0.48      0.41      0.38       857



### 3.1.2 Тиньк

In [126]:
y_pred_TCSG = train_predict_dt(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.3480
F1-score: 0.3475
AUC-ROC: 0.4885

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.44      0.40       202
   N (нейтр)       0.41      0.35      0.38       274
    H (рост)       0.19      0.19      0.19       116

    accuracy                           0.35       592
   macro avg       0.32      0.33      0.32       592
weighted avg       0.35      0.35      0.35       592



### 3.1.3 Газпром

In [128]:
y_pred_GAZP = train_predict_dt(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.3512
F1-score: 0.3622
AUC-ROC: 0.4556

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.35      0.34      0.34       285
   N (нейтр)       0.46      0.42      0.44       464
    H (рост)       0.07      0.11      0.09       108

    accuracy                           0.35       857
   macro avg       0.29      0.29      0.29       857
weighted avg       0.38      0.35      0.36       857



### 3.1.4 Лукойл

In [131]:
y_pred_LKOH = train_predict_dt(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.3092
F1-score: 0.2744
AUC-ROC: 0.4723

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.30      0.63      0.41       270
   N (нейтр)       0.35      0.19      0.24       388
    H (рост)       0.25      0.11      0.15       199

    accuracy                           0.31       857
   macro avg       0.30      0.31      0.27       857
weighted avg       0.31      0.31      0.27       857



### 3.1.5 Роснефть

In [134]:
y_pred_ROSN = train_predict_dt(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.3676
F1-score: 0.3611
AUC-ROC: 0.5253

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.35      0.58      0.44       291
   N (нейтр)       0.57      0.27      0.36       415
    H (рост)       0.20      0.23      0.21       151

    accuracy                           0.37       857
   macro avg       0.37      0.36      0.34       857
weighted avg       0.43      0.37      0.36       857



## Вывод

Не плохие результаты для самой базовой модели ) </br>
На разных акциях точность отличается

## 3.2 Регрессия

### 3.2.1 Сбер

In [141]:
y_pred_SBER = train_predict_logreg(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.3162
F1-score: 0.3153
AUC-ROC: 0.5020

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.32      0.40      0.36       269
   N (нейтр)       0.39      0.25      0.31       379
    H (рост)       0.25      0.32      0.28       209

    accuracy                           0.32       857
   macro avg       0.32      0.33      0.31       857
weighted avg       0.33      0.32      0.32       857



### 3.2.2 Тиньк

In [143]:
y_pred_TCSG = train_predict_logreg(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.4189
F1-score: 0.3393
AUC-ROC: 0.5721

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.37      0.91      0.53       202
   N (нейтр)       0.64      0.23      0.34       274
    H (рост)       0.00      0.00      0.00       116

    accuracy                           0.42       592
   macro avg       0.34      0.38      0.29       592
weighted avg       0.42      0.42      0.34       592



### 3.2.3 Газпром

In [146]:
y_pred_GAZP = train_predict_logreg(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.4352
F1-score: 0.4147
AUC-ROC: 0.5395

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.35      0.60      0.44       285
   N (нейтр)       0.57      0.44      0.50       464
    H (рост)       0.00      0.00      0.00       108

    accuracy                           0.44       857
   macro avg       0.31      0.34      0.31       857
weighted avg       0.43      0.44      0.41       857



### 3.2.4 Лукойл

In [148]:
y_pred_LKOH = train_predict_logreg(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.4026
F1-score: 0.3876
AUC-ROC: 0.5858

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.68      0.47       270
   N (нейтр)       0.64      0.36      0.46       388
    H (рост)       0.18      0.12      0.14       199

    accuracy                           0.40       857
   macro avg       0.39      0.38      0.36       857
weighted avg       0.44      0.40      0.39       857



### 3.2.5 Роснефть

In [150]:
y_pred_ROSN = train_predict_logreg(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.3956
F1-score: 0.3460
AUC-ROC: 0.5519

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.77      0.49       291
   N (нейтр)       0.55      0.27      0.36       415
    H (рост)       0.13      0.02      0.03       151

    accuracy                           0.40       857
   macro avg       0.35      0.35      0.29       857
weighted avg       0.41      0.40      0.35       857



## Вывод

Качество лучше по сравнению с деревом

## 3.3 Вот он, лес

### 3.3.1 Сбер

In [155]:
y_pred_SBER = train_predict_rf(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.3862
F1-score: 0.3656
AUC-ROC: 0.5472

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.70      0.47       269
   N (нейтр)       0.61      0.30      0.40       379
    H (рост)       0.20      0.13      0.16       209

    accuracy                           0.39       857
   macro avg       0.39      0.38      0.35       857
weighted avg       0.43      0.39      0.37       857



### 3.3.2 Тиньк

In [157]:
y_pred_TCSG = train_predict_rf(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.4105
F1-score: 0.3871
AUC-ROC: 0.5529

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.39      0.73      0.51       202
   N (нейтр)       0.64      0.25      0.36       274
    H (рост)       0.25      0.23      0.24       116

    accuracy                           0.41       592
   macro avg       0.43      0.40      0.37       592
weighted avg       0.48      0.41      0.39       592



### 3.3.3 Газпром

In [161]:
y_pred_GAZP = train_predict_rf(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.3886
F1-score: 0.3851
AUC-ROC: 0.4514

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.32      0.46      0.38       285
   N (нейтр)       0.51      0.42      0.46       464
    H (рост)       0.12      0.07      0.09       108

    accuracy                           0.39       857
   macro avg       0.32      0.32      0.31       857
weighted avg       0.40      0.39      0.39       857



### 3.3.4 Лукойл

In [163]:
y_pred_LKOH = train_predict_rf(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.3664
F1-score: 0.3091
AUC-ROC: 0.5080

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.33      0.83      0.47       270
   N (нейтр)       0.59      0.17      0.27       388
    H (рост)       0.36      0.11      0.17       199

    accuracy                           0.37       857
   macro avg       0.43      0.37      0.30       857
weighted avg       0.45      0.37      0.31       857



### 3.3.5 Роснефть

In [165]:
y_pred_ROSN = train_predict_rf(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.4224
F1-score: 0.3785
AUC-ROC: 0.5853

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.38      0.82      0.52       291
   N (нейтр)       0.62      0.26      0.37       415
    H (рост)       0.29      0.09      0.14       151

    accuracy                           0.42       857
   macro avg       0.43      0.39      0.34       857
weighted avg       0.48      0.42      0.38       857



## Вывод

Показывает лучшее качество по сравнению с деревом

## 3.4 Бустинг

### 3.4.1 Сбер

In [170]:
y_pred_SBER = train_predict_catboost(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.3676
F1-score: 0.3519
AUC-ROC: 0.5405

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.36      0.60      0.45       269
   N (нейтр)       0.49      0.34      0.40       379
    H (рост)       0.17      0.11      0.14       209

    accuracy                           0.37       857
   macro avg       0.34      0.35      0.33       857
weighted avg       0.37      0.37      0.35       857



### 3.4.2 Тиньк

In [173]:
y_pred_TCSG = train_predict_catboost(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.3682
F1-score: 0.3680
AUC-ROC: 0.5456

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.38      0.55      0.45       202
   N (нейтр)       0.57      0.27      0.37       274
    H (рост)       0.19      0.28      0.22       116

    accuracy                           0.37       592
   macro avg       0.38      0.37      0.35       592
weighted avg       0.43      0.37      0.37       592



### 3.4.3 Газпром

In [177]:
y_pred_GAZP = train_predict_catboost(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.4002
F1-score: 0.3997
AUC-ROC: 0.4698

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.31      0.38      0.34       285
   N (нейтр)       0.52      0.48      0.50       464
    H (рост)       0.14      0.10      0.12       108

    accuracy                           0.40       857
   macro avg       0.32      0.32      0.32       857
weighted avg       0.40      0.40      0.40       857



### 3.4.4 Лукойл

In [179]:
y_pred_LKOH = train_predict_catboost(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.3769
F1-score: 0.3204
AUC-ROC: 0.5666

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.34      0.85      0.48       270
   N (нейтр)       0.57      0.17      0.26       388
    H (рост)       0.44      0.15      0.22       199

    accuracy                           0.38       857
   macro avg       0.45      0.39      0.32       857
weighted avg       0.47      0.38      0.32       857



### 3.4.5 Роснефть

In [182]:
y_pred_ROSN = train_predict_catboost(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.4119
F1-score: 0.3787
AUC-ROC: 0.6097

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.37      0.76      0.50       291
   N (нейтр)       0.58      0.28      0.37       415
    H (рост)       0.27      0.11      0.16       151

    accuracy                           0.41       857
   macro avg       0.41      0.38      0.34       857
weighted avg       0.45      0.41      0.38       857



## Вывод

Бустинг показал лучшее качество на всех акциях кроме Роснефти по сравнению с лесом, даже на Сбере )

В данном сравнении лучше всего показал себя градиентный бустинг